In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 256, 7, 1, 0),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 1, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 128
lr = 0.002
z_dim = 100
epochs = 10

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)

In [ ]:
criterion = nn.BCELoss()

generator = Generator(z_dim).to(device)
discriminator = Discriminator().to(device)

optimizer_g = optim.Adam(
    generator.parameters(),
    lr=lr,
    betas=(0.5, 0.999)
)

optimizer_d = optim.Adam(
    discriminator.parameters(),
    lr=lr,
    betas=(0.5, 0.999)
)

In [ ]:
def show_images(fake):
    img = fake.detach().cpu()[:16]
    grid = img.view(16, 28, 28)

    fig, axes = plt.subplots(4, 4, figsize=(5, 5))

    for i, ax in enumerate(axes.flatten()):
        ax.imshow(grid[i], cmap="gray")
        ax.axis("off")

    plt.show()

In [ ]:
for epoch in range(epochs):

    for real, _ in dataloader:

        real = real.to(device)
        batch_size = real.size(0)

        # Generate fake images
        noise = torch.randn(batch_size, z_dim, 1, 1).to(device)
        fake = generator(noise)

        # Labels
        real_labels = torch.ones(batch_size, 1).to(device) * 0.9
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # Train Discriminator
        D_real = discriminator(real)
        loss_real = criterion(D_real, real_labels)

        D_fake = discriminator(fake.detach())
        loss_fake = criterion(D_fake, fake_labels)

        loss_D = loss_real + loss_fake

        optimizer_d.zero_grad()
        loss_D.backward()
        optimizer_d.step()

        # Train Generator
        output = discriminator(fake)
        g_loss = criterion(output, real_labels)

        optimizer_g.zero_grad()
        g_loss.backward()
        optimizer_g.step()

    show_images(fake)

    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Loss D: {loss_D.item():.4f} | "
        f"Loss G: {g_loss.item():.4f}"
    )

In [ ]:
# Extension Task: Generate Higher Resolution Images (64x64)

"""Objective:
Modify the GAN architecture to generate higher-resolution images (64x64) instead of the original 28x28 MNIST images.

Approach:
- Added extra ConvTranspose2D layers in Generator for progressive upsampling.
- Added extra Conv2D layers in Discriminator for handling larger images.
- Increased output resolution from 28x28 to 64x64."""

In [ ]:
class HighResGenerator(nn.Module):
    def __init__(self, z_dim):
        super().__init__()

        self.model = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 512, 4, 1, 0),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 1, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
class HighResDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            nn.Conv2d(256, 512, 4, 2, 1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2),

            nn.Flatten(),
            nn.Linear(512 * 4 * 4, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
high_res_generator = HighResGenerator(z_dim)

noise = torch.randn(1, z_dim, 1, 1)

generated_image = high_res_generator(noise)

print("Generated Image Shape:", generated_image.shape)

In [ ]:
## Extension Task Result

"""The GAN architecture was successfully modified to generate higher-resolution images.

Changes Made:
- Generator output increased from 28x28 to 64x64.
- Additional transposed convolution layers were added.
- Discriminator architecture was extended to process larger images."""

'The GAN architecture was successfully modified to generate higher-resolution images.\n\nChanges Made:\n- Generator output increased from 28x28 to 64x64.\n- Additional transposed convolution layers were added.\n- Discriminator architecture was extended to process larger images.'